# Legal Assistant Qwen2.5-3B QLoRA Trainer (fast GPU persistent)

Separate Colab notebook for finishing a 3B adapter quickly on a paid A100/L4 runtime. It keeps the hardened persistence path but removes the expensive mid-training eval and duplicate Hub push behavior that made the older 3B run crawl.

Run order:
1. In Colab, choose a premium GPU runtime first. Prefer A100. L4 is acceptable with the L4 profile. Use T4 only for smoke testing.
2. Run setup/login. Use a Hugging Face write token and keep it in the runtime environment as `HF_TOKEN`.
3. Optional: run `TRAINING_PROFILE = "PERSISTENCE_SMOKE"` first for a 5-10 minute proof that Drive and Hub saves work.
4. For the real paid run, use `A100_FAST_FINISH` on A100 or `L4_FAST_FINISH` on L4. These profiles use fixed `MAX_STEPS` so the run ends and saves a final adapter.
5. Do not leave the run unattended until `PERSISTENCE PREFLIGHT PASSED` and the first `adapter-checkpoints/checkpoint-*` folder appear on both Hugging Face and Google Drive.
6. If the early speed estimate says the run will exceed the profile budget, the notebook requests a save and clean stop instead of silently burning money.

This writes to a separate `-fast-lora` repo/path so it does not collide with the older 3B Colab run.


## 1. Install

In [ ]:
%pip install -q "transformers>=5.0.0" "datasets>=2.20.0" "accelerate>=0.33.0" \
    "peft>=0.12.0" "bitsandbytes>=0.43.0" "trl>=0.10.1" "huggingface_hub>=0.24.0" \
    "sentencepiece>=0.2.0" "protobuf>=4.25.0" "einops>=0.8.0"


## 2. Auth

In [ ]:
import os
from getpass import getpass
from huggingface_hub import login, whoami

if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass('Paste your HF token (write scope): ')
login(token=os.environ['HF_TOKEN'])
print('logged in as:', whoami(token=os.environ['HF_TOKEN'])['name'])

## 3. Configuration

In [ ]:
BASE_MODEL     = 'Qwen/Qwen2.5-3B-Instruct'
DATA_REPO      = 'tanziro/bd-legal-sft'
HF_OUTPUT_REPO = 'tanziro/bd-legal-qwen25-3b-fast-lora'

# Paid Colab target. Default is bounded so it produces an adapter quickly.
# Choices: PERSISTENCE_SMOKE, A100_FAST_FINISH, L4_FAST_FINISH, A100_FULL_FAST
TRAINING_PROFILE = 'A100_FAST_FINISH'

PROFILE_PRESETS = {
    'PERSISTENCE_SMOKE': {
        'MAX_LEN': 256, 'BATCH_SIZE': 1, 'GRAD_ACCUM': 16, 'EPOCHS': 1.0,
        'SAVE_STEPS': 5, 'EVAL_STEPS': 0, 'LORA_R': 8, 'LORA_ALPHA': 16,
        'SUBSET_TRAIN': 512, 'SUBSET_VAL': 64, 'MAX_STEPS': 10, 'TIME_BUDGET_HOURS': 0.5,
    },
    'A100_FAST_FINISH': {
        'MAX_LEN': 512, 'BATCH_SIZE': 4, 'GRAD_ACCUM': 8, 'EPOCHS': 1.0,
        'SAVE_STEPS': 25, 'EVAL_STEPS': 0, 'LORA_R': 16, 'LORA_ALPHA': 32,
        'SUBSET_TRAIN': 20000, 'SUBSET_VAL': 300, 'MAX_STEPS': 625, 'TIME_BUDGET_HOURS': 2.5,
    },
    'L4_FAST_FINISH': {
        'MAX_LEN': 512, 'BATCH_SIZE': 2, 'GRAD_ACCUM': 16, 'EPOCHS': 1.0,
        'SAVE_STEPS': 25, 'EVAL_STEPS': 0, 'LORA_R': 16, 'LORA_ALPHA': 32,
        'SUBSET_TRAIN': 16000, 'SUBSET_VAL': 300, 'MAX_STEPS': 500, 'TIME_BUDGET_HOURS': 4.0,
    },
    'A100_FULL_FAST': {
        'MAX_LEN': 512, 'BATCH_SIZE': 4, 'GRAD_ACCUM': 8, 'EPOCHS': 1.0,
        'SAVE_STEPS': 50, 'EVAL_STEPS': 0, 'LORA_R': 16, 'LORA_ALPHA': 32,
        'SUBSET_TRAIN': None, 'SUBSET_VAL': 500, 'MAX_STEPS': -1, 'TIME_BUDGET_HOURS': 8.0,
    },
}

profile = PROFILE_PRESETS[TRAINING_PROFILE]
MAX_LEN          = profile['MAX_LEN']
BATCH_SIZE       = profile['BATCH_SIZE']
GRAD_ACCUM       = profile['GRAD_ACCUM']
EPOCHS           = profile['EPOCHS']
SAVE_STEPS       = profile['SAVE_STEPS']
EVAL_STEPS       = profile['EVAL_STEPS']
LORA_R           = profile['LORA_R']
LORA_ALPHA       = profile['LORA_ALPHA']
SUBSET_TRAIN     = profile['SUBSET_TRAIN']
SUBSET_VAL       = profile['SUBSET_VAL']
MAX_STEPS        = profile['MAX_STEPS']
TIME_BUDGET_HOURS = profile['TIME_BUDGET_HOURS']

LEARNING_RATE    = 2e-4
WARMUP_RATIO     = 0.03
LORA_DROPOUT     = 0.05
SEED             = 42

# Refuse to burn a real fast run on a slow/free GPU. Smoke runs are allowed anywhere.
REQUIRE_FAST_GPU = TRAINING_PROFILE != 'PERSISTENCE_SMOKE'
FAST_GPU_MARKERS = ('A100', 'L4', 'H100', 'V100')

# Stop early if the first few optimizer steps predict another money-burning crawl.
ABORT_IF_PROJECTED_OVER_HOURS = TRAINING_PROFILE != 'PERSISTENCE_SMOKE'
PROJECTION_CHECK_STEPS = 10
MAX_PROJECTED_HOURS = TIME_BUDGET_HOURS * 1.25

OUTPUT_DIR       = '/content/legal-sft-qwen25-3b-fast-out'
FINAL_ADAPTER_DIR = '/content/legal-sft-qwen25-3b-fast-final-adapter'
FINAL_HUB_SUBFOLDER = 'final-adapter'

# Persistent backup. Keep this on unless you are intentionally running outside Colab.
BACKUP_TO_DRIVE = True
DRIVE_BACKUP_DIR = '/content/drive/MyDrive/legal-assistant-bd-legal-qwen25-3b-fast-lora'
DRIVE_FINAL_ADAPTER_DIR = DRIVE_BACKUP_DIR + '/final-adapter'

# Second persistence rail: verified Hub adapter checkpoints at every save.
HUB_BACKUP_EVERY_SAVE = True
HUB_CHECKPOINT_PREFIX = 'adapter-checkpoints'

# Do not use this for adapter-checkpoints/checkpoint-*. That path uses START_FROM_ADAPTER_SUBFOLDER.
RESUME_FROM_HUB  = False

# Continue from a saved adapter without needing full optimizer state.
# Example: START_FROM_ADAPTER_SUBFOLDER = 'adapter-checkpoints/checkpoint-150'
START_FROM_ADAPTER_REPO = HF_OUTPUT_REPO
START_FROM_ADAPTER_SUBFOLDER = None

print('training profile:', TRAINING_PROFILE, profile)


## 4. Pre-create the model repo (idempotent)

The fast notebook keeps generic Trainer Hub pushing disabled, but the repo must exist before the verified `upload_adapter_copy(...)` checkpoint uploads can write to it.


In [ ]:
from huggingface_hub import create_repo
create_repo(repo_id=HF_OUTPUT_REPO, repo_type='model', private=True, exist_ok=True)
print('model repo ready: https://huggingface.co/' + HF_OUTPUT_REPO)

## 5. Load + (optionally) subset the dataset

In [ ]:
from datasets import load_dataset
import random

ds = load_dataset(DATA_REPO, token=os.environ['HF_TOKEN'])
print('full:', ds)

def stratified_subset(d, n, key='task_type', seed=SEED):
    if n is None or len(d) <= n: return d
    buckets = {}
    for i, t in enumerate(d[key]):
        buckets.setdefault(t, []).append(i)
    rng = random.Random(seed)
    per = max(1, n // len(buckets))
    picks = []
    for t, idxs in buckets.items():
        rng.shuffle(idxs)
        picks.extend(idxs[:per])
    rng.shuffle(picks)
    return d.select(picks[:n])

ds['train'] = stratified_subset(ds['train'], SUBSET_TRAIN)
ds['validation'] = stratified_subset(ds['validation'], SUBSET_VAL)
print('subset:', ds)
from collections import Counter
print('train tasks:', Counter(ds['train']['task_type']))
print('val tasks:',   Counter(ds['validation']['task_type']))

## 6. Tokenize (loss masked to response only)

In [ ]:
from transformers import AutoTokenizer

SYSTEM_PROMPT = (
    'You are a Bangladesh legal research assistant. You are not a lawyer. '
    'Cite the official Laws of Bangladesh portal. '
    'Refuse if the retrieved evidence is insufficient.'
)

def render_prompt(row):
    return (f"<SYSTEM>{SYSTEM_PROMPT}</SYSTEM>\n"
            f"<INSTRUCTION>{row.get('instruction','')}</INSTRUCTION>\n"
            f"<CONTEXT>{row.get('context','')}</CONTEXT>\n"
            f"<RESPONSE>")

def render_full(row):
    return render_prompt(row) + f"{row.get('response','')}</RESPONSE>"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize(row):
    prompt = render_prompt(row); full = render_full(row)
    # Dynamic padding happens in the collator. Avoid max-length padding every row; it wastes paid GPU time.
    enc_full   = tokenizer(full, truncation=True, max_length=MAX_LEN, padding=False)
    enc_prompt = tokenizer(prompt, truncation=True, max_length=MAX_LEN, add_special_tokens=False)
    labels = list(enc_full['input_ids'])
    plen = min(len(enc_prompt['input_ids']), len(labels))
    for i in range(plen): labels[i] = -100
    pad = tokenizer.pad_token_id
    for i, t in enumerate(enc_full['input_ids']):
        if t == pad: labels[i] = -100
    enc_full['labels'] = labels
    return enc_full

ds_tok = ds.map(tokenize, remove_columns=ds['train'].column_names, num_proc=2, desc='tokenize')
print(ds_tok)


## 7. Load base model in 4-bit + LoRA

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb, device_map='auto', trust_remote_code=True)
model.config.use_cache = False
try:
    model = prepare_model_for_kbit_training(
        model,
        use_gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': False},
    )
except TypeError:
    # Older PEFT releases do not accept gradient_checkpointing_kwargs here.
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False)
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
model.enable_input_require_grads()
if START_FROM_ADAPTER_SUBFOLDER:
    print('continuing from adapter:', START_FROM_ADAPTER_REPO, START_FROM_ADAPTER_SUBFOLDER)
    model = PeftModel.from_pretrained(
        model,
        START_FROM_ADAPTER_REPO,
        subfolder=START_FROM_ADAPTER_SUBFOLDER,
        token=os.environ['HF_TOKEN'],
        is_trainable=True,
    )
else:
    model = get_peft_model(model, LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        bias='none', task_type='CAUSAL_LM',
        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    ))
model.print_trainable_parameters()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'trainable params: {trainable:,} / {total:,}')
assert trainable > 0, 'no trainable LoRA parameters; check target_modules/model architecture'


## 8. Train (with incremental push to the Hub every 200 steps)

If Colab disconnects mid-train, your latest checkpoint is still on the Hub. Re-running the notebook will resume from it.

In [ ]:
from transformers import Trainer, TrainingArguments, TrainerCallback
from huggingface_hub import HfApi, create_repo, upload_file, upload_folder
import os, traceback, inspect, json, time
import transformers, peft, accelerate

api = HfApi(token=os.environ['HF_TOKEN'])

def run_persistence_preflight():
    stamp = {'time': time.time(), 'repo': HF_OUTPUT_REPO, 'save_steps': SAVE_STEPS}
    if BACKUP_TO_DRIVE:
        drive_probe = os.path.join(DRIVE_BACKUP_DIR, 'drive-preflight.json')
        with open(drive_probe, 'w', encoding='utf-8') as f:
            json.dump(stamp, f, indent=2)
        assert os.path.exists(drive_probe), 'Drive preflight write failed'
        print('Drive preflight wrote:', drive_probe)

    local_probe = '/content/hub-persistence-preflight.json'
    with open(local_probe, 'w', encoding='utf-8') as f:
        json.dump(stamp, f, indent=2)
    create_repo(HF_OUTPUT_REPO, repo_type='model', private=True, exist_ok=True, token=os.environ['HF_TOKEN'])
    upload_file(
        path_or_fileobj=local_probe,
        path_in_repo='persistence-probes/latest.json',
        repo_id=HF_OUTPUT_REPO,
        repo_type='model',
        token=os.environ['HF_TOKEN'],
        commit_message='Persistence preflight probe',
    )
    remote_files = set(api.list_repo_files(HF_OUTPUT_REPO, repo_type='model'))
    assert 'persistence-probes/latest.json' in remote_files, 'Hub preflight upload failed'
    print('PERSISTENCE PREFLIGHT PASSED')

if BACKUP_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
        print('Drive backup dir:', DRIVE_BACKUP_DIR)
    except Exception:
        print('DRIVE BACKUP SETUP FAILED - refusing to train without persistent backup')
        traceback.print_exc()
        raise

run_persistence_preflight()

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'
if REQUIRE_FAST_GPU and not any(marker in gpu_name for marker in FAST_GPU_MARKERS):
    raise RuntimeError(
        f"TRAINING_PROFILE={TRAINING_PROFILE} expects a paid fast GPU ({FAST_GPU_MARKERS}), but Colab assigned {gpu_name}. "
        "Reconnect with a premium GPU, or set TRAINING_PROFILE='PERSISTENCE_SMOKE'."
    )
print('gpu accepted for profile:', gpu_name)

def verify_adapter_dir(path):
    files = set(os.listdir(path)) if os.path.isdir(path) else set()
    assert 'adapter_config.json' in files, f'adapter_config.json missing in {path}'
    assert ('adapter_model.safetensors' in files) or ('adapter_model.bin' in files), f'adapter weights missing in {path}'
    return files

def save_adapter_copy(model, target_dir, label):
    os.makedirs(target_dir, exist_ok=True)
    model.save_pretrained(target_dir, safe_serialization=True)
    tokenizer.save_pretrained(target_dir)
    files = verify_adapter_dir(target_dir)
    print(f'{label} saved:', target_dir, sorted(files))
    return files

def verify_remote_adapter(path_in_repo=''):
    prefix = (path_in_repo.strip('/') + '/') if path_in_repo else ''
    remote_files = set(api.list_repo_files(HF_OUTPUT_REPO, repo_type='model'))
    assert prefix + 'adapter_config.json' in remote_files, f'remote adapter_config.json missing at {prefix or "repo root"}'
    assert (prefix + 'adapter_model.safetensors' in remote_files) or (prefix + 'adapter_model.bin' in remote_files), f'remote adapter weights missing at {prefix or "repo root"}'
    return remote_files

def upload_adapter_copy(source_dir, path_in_repo, label):
    verify_adapter_dir(source_dir)
    create_repo(HF_OUTPUT_REPO, repo_type='model', private=True, exist_ok=True, token=os.environ['HF_TOKEN'])
    print('uploading adapter folder:', source_dir)
    print('to:', 'https://huggingface.co/' + HF_OUTPUT_REPO + '/tree/main/' + path_in_repo)
    for attempt in range(1, 5):
        try:
            upload_folder(
                repo_id=HF_OUTPUT_REPO,
                repo_type='model',
                folder_path=source_dir,
                path_in_repo=path_in_repo,
                token=os.environ['HF_TOKEN'],
                commit_message=f'{label} attempt {attempt}',
            )
            break
        except Exception as e:
            print(f'upload attempt {attempt} failed:', type(e).__name__, e)
            traceback.print_exc()
            time.sleep(5 * attempt)
    else:
        raise RuntimeError('all 4 upload attempts failed for ' + path_in_repo)
    verify_remote_adapter(path_in_repo)
    print(f'{label} verified on Hub at {path_in_repo}')

class WallClockStopCallback(TrainerCallback):
    def __init__(self, max_hours):
        self.max_seconds = None if not max_hours or max_hours <= 0 else max_hours * 3600
        self.started_at = None

    def on_train_begin(self, args, state, control, **kwargs):
        self.started_at = time.time()
        return control

    def on_step_end(self, args, state, control, **kwargs):
        if self.max_seconds and self.started_at and time.time() - self.started_at >= self.max_seconds:
            print(f'Time budget reached at step {state.global_step}; requesting save and clean stop.')
            control.should_save = True
            control.should_training_stop = True
        return control

class ProjectedTimeGuardCallback(TrainerCallback):
    def __init__(self, target_steps, check_steps, max_projected_hours, enabled=True):
        self.target_steps = max(1, target_steps)
        self.check_steps = max(1, check_steps)
        self.max_projected_seconds = None if not max_projected_hours else max_projected_hours * 3600
        self.enabled = enabled
        self.started_at = None

    def on_train_begin(self, args, state, control, **kwargs):
        self.started_at = time.time()
        return control

    def on_step_end(self, args, state, control, **kwargs):
        if not self.enabled or not self.started_at or state.global_step < self.check_steps:
            return control
        elapsed = time.time() - self.started_at
        seconds_per_step = elapsed / max(1, state.global_step)
        projected = seconds_per_step * self.target_steps
        print({
            'speed_check_step': state.global_step,
            'seconds_per_step': round(seconds_per_step, 2),
            'projected_hours': round(projected / 3600, 2),
            'max_projected_hours': round((self.max_projected_seconds or 0) / 3600, 2),
        })
        if self.max_projected_seconds and projected > self.max_projected_seconds:
            print('Projected runtime is over budget; requesting save and clean stop before more money burns.')
            control.should_save = True
            control.should_training_stop = True
        self.enabled = False
        return control

class DriveAdapterBackupCallback(TrainerCallback):
    def on_save(self, args, state, control, model=None, **kwargs):
        if not BACKUP_TO_DRIVE:
            return control
        if model is None:
            raise RuntimeError('Trainer did not provide model to Drive backup callback')
        checkpoint_dir = os.path.join(DRIVE_BACKUP_DIR, f'checkpoint-{state.global_step}')
        save_adapter_copy(model, checkpoint_dir, f'Drive adapter checkpoint step {state.global_step}')
        if HUB_BACKUP_EVERY_SAVE:
            path_in_repo = f'{HUB_CHECKPOINT_PREFIX}/checkpoint-{state.global_step}'
            upload_adapter_copy(checkpoint_dir, path_in_repo, f'Verified adapter-checkpoints/checkpoint-{state.global_step}')
        return control

class DynamicCausalLMCollator:
    def __init__(self, tokenizer, label_pad_token_id=-100):
        self.tokenizer = tokenizer
        self.label_pad_token_id = label_pad_token_id

    def __call__(self, features):
        labels = [feature.pop('labels') for feature in features]
        batch = self.tokenizer.pad(
            features,
            padding=True,
            pad_to_multiple_of=8,
            return_tensors='pt',
        )
        max_len = batch['input_ids'].shape[1]
        padded_labels = []
        for label in labels:
            padded_labels.append(label + [self.label_pad_token_id] * (max_len - len(label)))
        batch['labels'] = torch.tensor(padded_labels, dtype=torch.long)
        return batch

collator = DynamicCausalLMCollator(tokenizer)

# Detect resume-from-hub: if the model repo already has any 'checkpoint-*' folder,
# download the latest and resume from it.
resume_from = None
try:
    files = api.list_repo_files(HF_OUTPUT_REPO, repo_type='model')
    ckpts = sorted({f.split('/')[0] for f in files if f.startswith('checkpoint-')},
                   key=lambda s: int(s.split('-')[1]))
    if RESUME_FROM_HUB and ckpts:
        from huggingface_hub import snapshot_download
        latest = ckpts[-1]
        print('found checkpoint on Hub:', latest, '-> downloading to resume')
        local = snapshot_download(HF_OUTPUT_REPO, repo_type='model',
                                  allow_patterns=[latest+'/*'],
                                  local_dir=OUTPUT_DIR)
        resume_from = os.path.join(OUTPUT_DIR, latest)
        print('will resume from:', resume_from)
    elif ckpts:
        print('Hub checkpoints found but RESUME_FROM_HUB=False; starting a clean run:', ckpts[-3:])
except Exception as e:
    print('no resumable checkpoint:', e)

steps_per_epoch = (len(ds_tok['train']) + (BATCH_SIZE * GRAD_ACCUM) - 1) // (BATCH_SIZE * GRAD_ACCUM)
unbounded_train_steps = max(1, int(steps_per_epoch * EPOCHS))
total_train_steps = unbounded_train_steps
if MAX_STEPS and MAX_STEPS > 0:
    total_train_steps = min(unbounded_train_steps, MAX_STEPS)
expected_examples = total_train_steps * BATCH_SIZE * GRAD_ACCUM
warmup_steps = max(1, int(total_train_steps * WARMUP_RATIO)) if WARMUP_RATIO else 0
eval_strategy = 'steps' if EVAL_STEPS and EVAL_STEPS > 0 else 'no'
print('step plan:', {'unbounded_steps': unbounded_train_steps, 'target_steps': total_train_steps, 'expected_examples': expected_examples, 'max_steps': MAX_STEPS})

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_steps=warmup_steps,
    weight_decay=0.0,
    logging_steps=10,
    eval_strategy=eval_strategy,
    eval_steps=EVAL_STEPS if eval_strategy == 'steps' else None,
    save_strategy='steps',
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    max_steps=MAX_STEPS,
    seed=SEED,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    optim='paged_adamw_8bit',
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    tf32=torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8,
    report_to=[],
    # The verified upload_adapter_copy(...) path handles Hub persistence.
    # Keeping Trainer's generic push disabled avoids duplicate checkpoint uploads.
    push_to_hub=False,
)

trainer_kwargs = dict(
    model=model, args=args,
    train_dataset=ds_tok['train'],
    eval_dataset=ds_tok['validation'],
    data_collator=collator,
    callbacks=(
        ([DriveAdapterBackupCallback()] if BACKUP_TO_DRIVE else [])
        + [ProjectedTimeGuardCallback(total_train_steps, PROJECTION_CHECK_STEPS, MAX_PROJECTED_HOURS, ABORT_IF_PROJECTED_OVER_HOURS)]
        + [WallClockStopCallback(TIME_BUDGET_HOURS)]
    ),
)
if 'processing_class' in inspect.signature(Trainer.__init__).parameters:
    trainer_kwargs['processing_class'] = tokenizer
else:
    trainer_kwargs['tokenizer'] = tokenizer
trainer = Trainer(**trainer_kwargs)
print('versions:', {'transformers': transformers.__version__, 'peft': peft.__version__, 'accelerate': accelerate.__version__, 'torch': torch.__version__})
print('gpu:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
print('train config:', {'profile': TRAINING_PROFILE, 'base_model': BASE_MODEL, 'rows_train': len(ds_tok['train']), 'rows_val': len(ds_tok['validation']), 'max_len': MAX_LEN, 'batch_size': BATCH_SIZE, 'grad_accum': GRAD_ACCUM, 'target_steps': total_train_steps, 'expected_examples': expected_examples, 'save_steps': SAVE_STEPS, 'eval_steps': EVAL_STEPS, 'max_steps': MAX_STEPS, 'time_budget_hours': TIME_BUDGET_HOURS, 'max_projected_hours': MAX_PROJECTED_HOURS, 'warmup_steps': warmup_steps, 'resume_from': resume_from})
try:
    train_result = trainer.train(resume_from_checkpoint=resume_from)
    print('train metrics:', train_result.metrics)
except Exception:
    print('TRAINING FAILED - full traceback follows')
    traceback.print_exc()
    raise
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Save a clean adapter-only folder for the final Hub push.
trainer.model.save_pretrained(FINAL_ADAPTER_DIR, safe_serialization=True)
tokenizer.save_pretrained(FINAL_ADAPTER_DIR)
verify_adapter_dir(FINAL_ADAPTER_DIR)
print('final adapter files:', sorted(os.listdir(FINAL_ADAPTER_DIR)))

if BACKUP_TO_DRIVE:
    save_adapter_copy(trainer.model, DRIVE_FINAL_ADAPTER_DIR, 'Drive final adapter')
    verify_adapter_dir(DRIVE_FINAL_ADAPTER_DIR)

## 9. Forced final push (retries, never silent)

In [ ]:
import os

if (not os.path.isdir(FINAL_ADAPTER_DIR)) and BACKUP_TO_DRIVE and os.path.isdir(DRIVE_FINAL_ADAPTER_DIR):
    print('local final adapter missing; pushing Drive final adapter instead')
    FINAL_ADAPTER_DIR = DRIVE_FINAL_ADAPTER_DIR

local_files = set(os.listdir(FINAL_ADAPTER_DIR))
print('pushing clean adapter folder:', FINAL_ADAPTER_DIR)
print('local adapter files:', sorted(local_files))
assert 'adapter_config.json' in local_files, 'local adapter_config.json missing; training did not save adapter'
assert ('adapter_model.safetensors' in local_files) or ('adapter_model.bin' in local_files), 'local adapter weights missing; training did not save adapter'

upload_adapter_copy(FINAL_ADAPTER_DIR, FINAL_HUB_SUBFOLDER, 'Final clean adapter push')
print('adapter verified on Hub: https://huggingface.co/' + HF_OUTPUT_REPO + '/tree/main/' + FINAL_HUB_SUBFOLDER)

## 10. Hardened sanity check

Reload the adapter **from the Hub** (not the local folder), run a known prompt, and **assert** the citation URL appears. If this fails you'll see a clear error instead of a misleading-looking output.

In [ ]:
from peft import PeftModel
import torch, gc

del model, trainer
gc.collect(); torch.cuda.empty_cache()

base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb,
                                            device_map='auto', trust_remote_code=True)
infer = PeftModel.from_pretrained(base, HF_OUTPUT_REPO, token=os.environ['HF_TOKEN'])
infer.eval()

test = {'instruction': 'What does section 302 of the Penal Code, 1860 provide? Quote the operative text and cite the source.',
        'context': ''}
enc = tokenizer(render_prompt(test), return_tensors='pt').to(infer.device)
out = infer.generate(**enc, max_new_tokens=320, do_sample=False,
                     pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
txt = tokenizer.decode(out[0], skip_special_tokens=True)
print('--- adapter output ---')
print(txt)
print('---')
assert 'bdlaws.minlaw.gov.bd' in txt or 'http' in txt, \
    'CITATION CHECK FAILED - adapter did not emit a source URL. Retrain or investigate dataset coverage.'
print('citation check: OK')
